Step 1: Data Loading and Initial Exploration 

Load all datasets using Pandas  

Inspect structure using .head(), .info(), .describe()  

Identify primary and foreign keys  

Understand relationships between tables  

In [ ]:
import pandas as pd
#=========================================
# 1.Load all the datasets using Pandas
# 2.Inspect datasets
#=========================================

#customers
#===============================================================
customers = pd.read_csv('./python_project_aiml_logicmojo_dataset/customers.csv')

#customers.head()
#customers.info()
#customers.describe()

#primary key - customer_id 
#foreign key - customer_zip_code_prefix
#===============================================================

#location
#===============================================================
location = pd.read_csv('./python_project_aiml_logicmojo_dataset/location.csv')

#location.head()
#location.info()
#location.describe()

#primary key - geolocation_zip_code_prefix + lat + long (This is still not sufficient)
#foreign key - No foreign key 
#===============================================================

#order_item
#===============================================================
order_item = pd.read_csv('./python_project_aiml_logicmojo_dataset/order_item.csv')

#order_item.head()
#order_item.info()
#order_item.describe()

#primary key - order_id + order_item_id
#foreign key - product_id, seller_id 
#===============================================================

#orders
#===============================================================
orders = pd.read_csv('./python_project_aiml_logicmojo_dataset/orders.csv')

#orders.head()
#orders.info()
#orders.describe()

#primary key - order_id 
#foreign key - customer_id 
#===============================================================

#payments
#===============================================================
payments = pd.read_csv('./python_project_aiml_logicmojo_dataset/payments.csv')

#payments.head()
#payments.info()
#payments.describe()

#primary key - order_id + payment_sequential
#foreign key - No foreign key 
#===============================================================

#products
#===============================================================
products = pd.read_csv('./python_project_aiml_logicmojo_dataset/products.csv')

#products.head()
#products.info()
#products.describe()

#primary key - product_id
#foreign key - product_category_name
#===============================================================

#reviews
#===============================================================
reviews = pd.read_csv('./python_project_aiml_logicmojo_dataset/reviews.csv')

#reviews.head()
#reviews.info()
#reviews.describe()

#primary key - review_id + order_id
#foreign key - order_id
#===============================================================

#sellers
#===============================================================
sellers = pd.read_csv('./python_project_aiml_logicmojo_dataset/sellers.csv')

#sellers.head()
#sellers.info()
#sellers.describe()

#primary key - seller_id
#foreign key - seller_zip_code_prefix
#===============================================================

#category_translation
#===============================================================
category = pd.read_csv('./python_project_aiml_logicmojo_dataset/category_translation.csv')

#category.head()
#category.info()
#category.describe()

#primary key - product_category_name
#foreign key - No foreign key
#===============================================================


Step 2: Data Cleaning and Preprocessing 

Handle missing values appropriately  

Remove duplicate records  

Convert date columns to datetime format  

Validate data types and ranges  

Standardize column names if required  

In [ ]:
#Missing Values

#customers.isna().sum()
#location.isna().sum()
#order_item.isna().sum()

#orders.isna().sum()   
#orders# has missing values for order_approved_at[160],order_delivered_carrier_date[1783],order_delivered_customer_date[2965] 

#Fill the missing values of datetime type with NaT indicator 
orders["order_approved_at"] = pd.to_datetime(orders["order_approved_at"], errors="coerce")

#Check what is the order status of orders with missing approval timestamp
#orders[orders["order_approved_at"].isna()]["order_status"].value_counts()

#For status with delivered status, indicates that there is some quality issue with data. 
#If this must me investigated then a flag for missing data can be created. But for this project, this is not going to be created. 
#canceled     141
#delivered     14
#created        5

#Create a flag for approved orders 
orders["is_approved"] = orders["order_approved_at"].notna()

orders["order_delivered_carrier_date"] = pd.to_datetime(orders["order_delivered_carrier_date"], errors="coerce")

#Check what is the order status of orders with missing order_delivered_carrier_date 
#orders[orders["order_delivered_carrier_date"].isna()]["order_status"].value_counts()
#unavailable    609
#canceled       550
#invoiced       314
#processing     301
#created          5
#approved         2
#delivered        2

#For status with delivered status, indicates that there is some quality issue with data. 
#If this must me investigated then a flag for missing data can be created. But for this project, this is not going to be created. 

#Create a flag for order_delivered_carrier_date orders 
orders["is_order_delivered_carrier_date"] = orders["order_delivered_carrier_date"].notna()

orders["order_delivered_customer_date"] = pd.to_datetime(orders["order_delivered_customer_date"], errors="coerce")

#Check what is the order status of orders with missing order_delivered_customer_date 
#orders[orders["order_delivered_customer_date"].isna()]["order_status"].value_counts()
#shipped        1107
#canceled        619
#unavailable     609
#invoiced        314
#processing      301
#delivered         8
#created           5
#approved          2

#For status with delivered status, indicates that there is some quality issue with data. 
#If this must me investigated then a flag for missing data can be created. But for this project, this is not going to be created. 

#Create a flag for is_order_delivered_customer_date orders 
orders["is_order_delivered_customer_date"] = orders["order_delivered_customer_date"].notna()

#Feature Engineer - Delivery time (order purchase to delivery date)
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"], errors="coerce")
orders["delivery_time_days"] = (
    orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]).dt.days

#payments.isna().sum()

#products.isna().sum()
#products# has missing values for product_category_name[610],product_name_lenght[610],product_description_lenght[610],"[610]
#product_weight_g[2],product_length_cm[2],product_height_cm[2],product_width_cm[2]

#Check to see if product meta data is missing
#products[products["product_name_lenght"].isna()][
#    ["product_id", "product_category_name", "product_name_lenght","product_description_lenght","product_photos_qty"]
#].head(15)

#If a product_id exists by the product meta data is missing, then it could be due to product catalog not being complete.
#Hence we can fill the missing data with "unknown"

#Find percentage of missing product category name data
#products["product_category_name"].isna().mean() * 100

#Flag missing product category name
products["missing_product_category_name"] = products["product_category_name"].isna()

#Fill the missing values with "unknown"
products["product_category_name"] = products["product_category_name"].fillna("unknown")

#reviews.isna().sum()
#reviews# has missing values for review_comment_title,review_comment_message
#This means a score was given but no textual comments were provided. This usually happens. 

#Here, a combined flag is stored, that is if either of review title or comment exists, then this flag is true.
#Fill the missing values with empty string, so that the string length operators can work on them
reviews["review_comment_title"] = reviews["review_comment_title"].fillna("")
reviews["review_comment_message"] = reviews["review_comment_message"].fillna("")

reviews["has_review_text"] = (reviews["review_comment_title"].str.len() > 0) | (reviews["review_comment_message"].str.len() > 0)

#sellers.isna().sum()
#category.isna().sum()

#Remove Duplicate Records
#customers["customer_id"].duplicated().sum()

#customers.groupby("customer_unique_id")["customer_id"].nunique().sort_values(ascending=False).head()

#The duplicated() output below indicate multiple zip code prefix that appear to be duplicates. 
#We cannot remove duplicates. For location the its tricky to get unique key even if we combine the zip code prefix and lat/long. 

#location.duplicated(
#    subset=[
#        "geolocation_zip_code_prefix",
#        "geolocation_lat",
#        "geolocation_lng"
#    ]).sum()

#We need to create a new clean dataframe, with a unique zip code. We group the rows with same lat/lng and keep the mean and retain 
#most occuring city and state names. This is because the city names could not be consistent

#location.groupby("geolocation_zip_code_prefix")["geolocation_city"].nunique().sort_values(ascending=False).head()
#location.loc[location["geolocation_zip_code_prefix"] == 13457, 'geolocation_city']

location_clean = (
    location
    .groupby("geolocation_zip_code_prefix", as_index=False)
    .agg({
        "geolocation_lat": "mean",
        "geolocation_lng": "mean",
        "geolocation_city": lambda x: x.mode().iloc[0],
        "geolocation_state": lambda x: x.mode().iloc[0]
    })
)

#location_clean["geolocation_zip_code_prefix"].is_unique

#order_item[order_item.duplicated(subset=["order_id", "order_item_id"], keep=False)].sum()

#orders["order_id"].duplicated().sum()

#payments.duplicated(subset=["order_id","payment_sequential"]).sum()

#products["product_id"].duplicated().sum()

#reviews.duplicated(subset=["review_id","order_id"]).sum()
#This also works
#reviews[["review_id","order_id"]].duplicated().sum()
#print(type(reviews[["review_id","order_id"]]))
#print(type(reviews.duplicated(subset=["review_id","order_id"])))

#sellers["seller_id"].duplicated().sum()

#category["product_category_name"].duplicated().sum()
#category["product_category_name"].isna().sum()

#Convert date columns to datetime format
order_item["shipping_limit_date"] = pd.to_datetime(order_item["shipping_limit_date"], errors="coerce")
#order_item.dtypes

orders["order_estimated_delivery_date"] = pd.to_datetime(orders["order_estimated_delivery_date"], errors="coerce")
#orders.dtypes

reviews["review_creation_date"] = pd.to_datetime(reviews["review_creation_date"], errors="coerce")

#Validate data types and ranges

#price_conv = pd.to_numeric(order_item["price"], errors="coerce")
#price_inv = order_item["price_conv".isna() & order_item["price"].notna()]
#price_inv.sum()

#seq_conv = pd.to_numeric(payments["payment_sequential"], errors="coerce")
#invalid_payment_sequential = payments[seq_conv.isna() & payments["payment_sequential"].notna()]
#invalid_payment_sequential.sum()

#Standardize column names if required

#payments["payment_type"].unique()
#orders["order_status"].unique()

products = products.rename(columns={
    "product_name_lenght": "product_name_length"
})

products = products.rename(columns={
    "product_description_lenght": "product_description_length"
})

#products.info()

Step 3: Data Integration (Critical Component) 

You must construct a Master Dataset by merging multiple tables. 

Recommended sequence: 

orders + customers  

orders + order_items  

order_items + products  

orders + payments  

orders + reviews  

order_items + sellers 

products + category_translation  

Final output: 
A single consolidated dataset representing a unified business view 

In [ ]:
# Merger orders + customers
orders_master = orders.merge(
    customers,
    on="customer_id",
    how="left"
)


# Merge orders + order_item
orders_master = orders_master.merge(
    order_item,
    on="order_id",
    how="left"
)

# Merge order_items + products
orders_master = orders_master.merge(
    products,
    on="product_id",
    how="left"
)

# Merge orders + payments
orders_master = orders_master.merge(
    payments,
    on="order_id",
    how="left"
)

# Merge orders + reviews
orders_master = orders_master.merge(
    reviews,
    on="order_id",
    how="left"
)

# Merge order_items + sellers 
orders_master = orders_master.merge(
    sellers,
    on="seller_id",
    how="left"
)

# Merge products + category_translation  
orders_master = orders_master.merge(
    category,
    on="product_category_name",
    how="left"
)

In [ ]:
orders_master.info()

Step 4: Feature Engineering 

Create meaningful features such as: 

Total order value (aggregated from order_items or payments)  

Delivery time (order purchase to delivery date)  

Number of items per order  

Customer purchase frequency  

Customer lifetime value (basic approximation)  

Average order value per customer  

In [ ]:
#Total order value (aggregated from order_items or payments)

order_total_payments = (
    payments
    .groupby("order_id", as_index=False)
    .agg(
        total_order_paid_value=("payment_value", "sum"),
        payment_count=("payment_sequential", "count"),
        payment_type_count=("payment_type", "nunique"),
        max_installments=("payment_installments", "max")
    )
)
orders_master = orders_master.merge(
    order_total_payments,
    on="order_id",
    how="left"
)

#Number of items per order

items_per_order = (
    orders_master
    .groupby("order_id")["order_item_id"]
    .count()
    .reset_index(name="items_per_order")
)

orders_master = orders_master.merge(items_per_order, on="order_id", how="left")
orders_master["items_per_order"] = orders_master["items_per_order"].fillna(0).astype(int)

#Customer purchase frequency
purchase_frequency = (
    orders_master
    .groupby("customer_unique_id")["order_id"]
    .nunique()
    .reset_index(name="customer_purchase_frequency")
)

orders_master = orders_master.merge(
    purchase_frequency,
    on="customer_unique_id",
    how="left"
)

#Customer lifetime value (basic approximation)

customer_lifetime_value = (
    orders_master[orders_master["order_status"] == "delivered"]
    .groupby("customer_unique_id")["price"]
    .sum()
    .reset_index(name="customer_lifetime_value")
)

orders_master = orders_master.merge(
    customer_lifetime_value,
    on="customer_unique_id",
    how="left"
)

#Average order value per customer

customer_avg_value = (
    orders_master[orders_master["order_status"] == "delivered"]
    .groupby(["customer_unique_id", "order_id"], as_index=False)
    .agg(order_value=("price", "sum"))
    .groupby("customer_unique_id", as_index=False)
    .agg(customer_avg_value=("order_value", "mean"))
)

orders_master = orders_master.merge(
    customer_avg_value,
    on="customer_unique_id",
    how="left"
)

In [ ]:
orders_master[["order_id","payment_value"]].head()

In [ ]:
orders_master[["order_id","items_per_order"]].head()

In [ ]:
orders_master[["order_id","delivery_time_days"]].head()

In [ ]:
orders_master[["customer_unique_id","customer_purchase_frequency"]].head()

In [ ]:
orders_master[["customer_unique_id","customer_lifetime_value"]].head()

In [ ]:
orders_master[["customer_unique_id","customer_avg_value"]].head()

Step 5: Exploratory Data Analysis (EDA) 

Perform structured analysis across the following dimensions: 


Customer Analysis 

New vs repeat customers  

High-value vs low-value customers  

Geographic distribution of customers  

 

Revenue and Order Analysis 

Monthly revenue trends  

Order volume trends  

Peak sales periods  

 

Product Analysis 

Top-selling product categories  

Revenue contribution by category  

Product demand distribution  

 

Seller Analysis 

Top-performing sellers  

Seller contribution to revenue  

Seller distribution  

 

Review and Satisfaction Analysis 

Distribution of review scores  

Relationship between delivery time and ratings  

Identification of dissatisfaction patterns  

In [ ]:
import numpy as np
#Find old and new customers - if number of orders for the customer is 1 then treat as new customer
#If number or orders > 1 then existing or repeat customer
customer_order_freq = (
    orders_master.groupby("customer_unique_id")
      .agg(
          order_count=("order_id", "nunique"),
      )
      .reset_index()
)
customer_order_freq["customer_type"] = customer_order_freq["order_count"].apply(lambda x: "Repeat" if x > 1 else "New")
customer_order_freq.groupby("customer_type")["order_count"].sum()

#Find the 75th percentile for customer_lifetime_value and treat the customers whose value is >= 75th as High Value. Rest are Low Value
qualify = orders_master["customer_lifetime_value"].quantile(0.75)

orders_master["customer_value"] = np.where(orders_master["customer_lifetime_value"] >= qualify, "High", "Low")
orders_master.groupby("customer_value")["customer_value"].count()

#Merge locations table with orders master

customer_locations = location_clean.rename(columns={
    "geolocation_zip_code_prefix": "customer_zip_code_prefix",
    "geolocation_city": "customerGeolocation_city",
    "geolocation_state": "customerGeolocation_state"
})

seller_locations = location_clean.rename(columns={
    "geolocation_zip_code_prefix": "seller_zip_code_prefix",
    "geolocation_city": "sellerGeolocation_city",
    "geolocation_state": "sellerGeolocation_state"
})

orders_master = (
    orders_master
    .merge(customer_locations, on="customer_zip_code_prefix", how="left")
    .merge(seller_locations, on="seller_zip_code_prefix", how="left")
)

customer_dist_by_state = (
    orders_master
    .groupby("customerGeolocation_state")["customer_unique_id"]
    .nunique()
    .reset_index(name="customer_count")
    .sort_values("customer_count", ascending=False)
) 

In [ ]:
customer_dist_by_state

In [ ]:
#Revenue and Order Analysis

#Monthly revenue trends
monthly_revenue = (
    orders_master
    .assign(month=orders_master["order_purchase_timestamp"].dt.to_period("M"))
    .groupby("month")["total_order_paid_value"]
    .sum()
    .reset_index(name="monthly_revenue")
)

print(monthly_revenue)

#Order volume trends
monthly_volume = (
    orders_master
    .assign(month=orders_master["order_purchase_timestamp"].dt.to_period("M"))
    .groupby("month")["order_id"]
    .count()
    .reset_index(name="monthly_volume")
)

print(monthly_volume)

#Peak sales periods
sales_peak_month = (
    orders_master
    .assign(month=orders_master["order_purchase_timestamp"].dt.to_period("M"))
    .groupby("month")["total_order_paid_value"]
    .sum()
    .reset_index(name="total_sales")
    .sort_values("total_sales", ascending=False)
)

print(sales_peak_month)

Product Analysis

Top-selling product categories

Revenue contribution by category

Product demand distribution

Seller Analysis

Top-performing sellers

Seller contribution to revenue

Seller distribution

Review and Satisfaction Analysis

Distribution of review scores

Relationship between delivery time and ratings

Identification of dissatisfaction patterns

In [ ]:
#Product Analysis

#Top-selling product categories
top_selling_products = (
    orders_master
    .groupby("product_category_name")["order_item_id"]
    .count()
    .reset_index(name="products_sold")
    .sort_values("products_sold", ascending=False)
)
print(top_selling_products)

#Revenue contribution by category
top_revenue_products = (
    orders_master
    .groupby("product_category_name")["price"]
    .sum()
    .reset_index(name="total_sales")
    .sort_values("total_sales", ascending=False)
)
print(top_revenue_products)

#Product demand distribution
#Based on product included in the order
product_order_demand = (
    orders_master
    .groupby(["product_id", "product_category_name"])["order_id"]
    .nunique()
    .reset_index(name="order_count")
    .sort_values("order_count", ascending=False)
)
print(product_order_demand)

In [ ]:
#Seller Analysis

#Top-performing sellers
top_seller = (
    orders_master
    .groupby("seller_id")["order_item_id"]
    .count()
    .reset_index(name="items_sold")
    .sort_values("items_sold", ascending=False)
)
print(top_seller)

#Seller contribution to revenue
top_seller_revenue = (
    orders_master
    .groupby("seller_id")["price"]
    .sum()
    .reset_index(name="total_sales")
    .sort_values("total_sales", ascending=False)
)
print(top_seller_revenue)

#Seller distribution
seller_demand = (
    orders_master
    .groupby(["seller_id"])["order_id"]
    .nunique()
    .reset_index(name="order_count")
    .sort_values("order_count", ascending=False)
)
print(seller_demand)

In [ ]:
#Review and Satisfaction Analysis

#Distribution of review scores
review_scores = (
    orders_master["review_score"]
    .value_counts()
    .sort_index()
    .reset_index()
)
print(review_scores)

#Relationship between delivery time and ratings
orders_master["review_score"] = pd.to_numeric(
    orders_master["review_score"],
    errors="coerce"
)

delivery_time_rating = (
    orders_master
    .dropna(subset=["delivery_time_days", "review_score"])
    .groupby("delivery_time_days")
    .agg(
        avg_review_score = ("review_score", "mean"),
        order_count = ("order_id", "nunique")
    )
    .reset_index()
    .sort_values("delivery_time_days")
)
print(delivery_time_rating)    

#Identification of dissatisfaction patterns
orders_master["is_dissatisfied"] = orders_master["review_score"].isin([1, 2])
#dissatisfied by product with rating 1 or 2 - this is a suggestive than assertive
dissatisfaction_by_product = (
    orders_master
    .groupby(["product_id", "product_category_name"])
    .agg(
        total_orders=("order_id", "nunique"),
        dissatisfied_orders=("is_dissatisfied", "sum"),
        avg_review_score=("review_score", "mean")
    )
    .reset_index()
)
print(dissatisfaction_by_product)

Step 6: Data Visualization 

Use Matplotlib and Seaborn to create: 

Time series plots (sales trends)  

Bar charts (category performance)  

Histograms (distribution analysis)  

Box plots (outlier detection)  

Heatmaps (correlation analysis)  

All visualizations must be clearly labeled and interpretable. 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

#Time series plots (sales trends)

orders_master["order_day"] = orders_master["order_purchase_timestamp"].dt.date
orders_master["order_month"] = orders_master["order_purchase_timestamp"].dt.to_period("M").astype(str)
orders_master["order_year"] = orders_master["order_purchase_timestamp"].dt.year

#*-------------- Daily Trend --------------------*
daily_sales = (
    orders_master.groupby("order_day")
      .agg(
          total_sales=("total_order_paid_value", "sum"),
          total_orders=("order_id", "count")
      )
      .reset_index()
)

plt.figure(figsize=(12, 5))
sns.lineplot(data=daily_sales, x="order_day", y="total_sales", marker="o")
#sns.barplot(data=daily_sales, x="order_day", y="total_sales")
plt.title("Daily Sales Trend")
plt.xlabel("Order Date")
plt.ylabel("Total Sales")
plt.xticks(rotation=45)
#plt.grid(True)
plt.tight_layout()
plt.show()

#*-------------- Monthly Trend --------------------*
monthly_sales = (
    orders_master.groupby("order_month")
      .agg(
          total_sales=("total_order_paid_value", "sum"),
          total_orders=("order_id", "count")
      )
      .reset_index()
)

plt.figure(figsize=(12, 5))
sns.barplot(data=monthly_sales, x="order_month", y="total_sales")
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Total Sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

#Bar charts (category performance)

#*-------------- Category Performance --------------------*
category_sales = (
    orders_master.groupby("product_category_name")
      .agg(
          total_sales=("price", "sum"),
          total_orders=("order_id", "count")
      )
      .reset_index()
      .sort_values("total_sales", ascending=False)
)

cat_top_10 = category_sales.head(10)

plt.figure(figsize=(12, 6))

sns.barplot(
    data=cat_top_10,
    x="product_category_name",
    y="total_sales"
)

plt.title("Top Categories by Sales")
plt.xlabel("Category")
plt.ylabel("Total Sales")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

#Histograms (distribution analysis)

#*-------------- Order Performance --------------------*
plt.figure(figsize=(10, 6))

sns.histplot(
    data=orders_master,
    x="total_order_paid_value",
    bins=15,
    kde=True,
    color="red"
)

plt.title("Order Value Distribution")
plt.xlabel("Order Amount")
plt.ylabel("Number of Orders")
plt.tight_layout()
plt.show()

#Box plots (outlier detection)
plt.figure(figsize=(10, 5))

sns.boxplot(
    data=orders_master,
    x="total_order_paid_value",
    color="red"
)

plt.title("Order Amount Outlier Detection")
plt.xlabel("Total Amount")
plt.tight_layout()
plt.show()

#Heatmaps (correlation analysis)
numeric_cols = [
    "total_order_paid_value",
    "max_installments",
    "payment_value"
]
corr_matrix = orders_master[numeric_cols].corr()

# ----------------Create heatmap-----------------

plt.figure(figsize=(10, 6))

sns.heatmap(
    corr_matrix,
    annot=True,
    cmap="coolwarm",
    linewidths=0.5
)

plt.title("Correlation Heatmap - Orders Master")
plt.tight_layout()
plt.show()